# Australian education providers for overseas students (CRICOS)

Two runs of [CRICOS Register Lookup (Australia)](https://apify.com/yappman/cricos-register):

1. **Search** the register for providers in one state, 10 rows, and count how many have a name shown.
2. **Look up** two of those providers by CRICOS provider code and compare their course counts.

Contains data from the Commonwealth Register of Institutions and Courses for Overseas Students (CRICOS), Department of Education, Australian Government, licensed under Creative Commons Attribution 2.5 Australia (CC BY 2.5 AU). This Actor filters and reshapes the data; changes were made.

Not affiliated with or endorsed by the Department of Education.

**Cost at Apify Free-tier prices:** at most **$0.024** in total.

| Run | Rows | Unit price | Most it can cost |
|---|---|---|---|
| Search, `maxItems` 10 | 10 | $0.002 | $0.02 |
| Lookup, 2 codes | 2 | $0.002 | $0.004 |

There is no start fee. The price is $0.002 a row on every plan.

More data: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
import time
from datetime import timedelta
from decimal import Decimal
from pathlib import Path

import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)
pd.set_option("display.html.use_mathjax", False)  # show third-party text as text

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)  # git-ignored: nothing you collect is committed


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped twice: maxItems in run_input limits the rows saved (search mode), and
    max_total_charge_usd stops the run if its charges would pass the cap.
    The run times out after 10 minutes.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
        run_timeout=timedelta(minutes=10),
    )
    if run is None or run.status != "SUCCEEDED":
        raise RuntimeError(f"{actor_id} did not succeed: {None if run is None else run.status}")
    print(f"{actor_id}: run {run.id}")
    # Charges settle a few seconds after the run ends, so read the record again before printing them.
    time.sleep(5)
    settled = client.run(run.id).get()
    print("Charged events:", settled.charged_event_counts)
    return run

## 1. Search the register

`search` returns the providers that pass the filters, sorted by provider code. A provider's name is left out when
the Actor's name rule cannot tell an organisation from a person (`nameWithheld` is then `true`), so the table below
shows the provider code and the register's own fields, and a name only where one is given.

In [ ]:
ACTOR = "yappman/cricos-register"

# Free tier: 10 rows x $0.002 = $0.02, no start fee
search_run = run_actor(
    ACTOR,
    {"mode": "search", "states": ["NSW"], "maxItems": 10},
    max_charge_usd=0.03,
)
providers = pd.DataFrame(client.dataset(search_run.default_dataset_id).iterate_items())
summary = client.key_value_store(search_run.default_key_value_store_id).get_record("RUN_SUMMARY")["value"]
print("snapshot", summary.get("snapshotDate"), "| matched", summary.get("matched"), "| returned", summary.get("returned"),
      "| names withheld", summary.get("namesWithheld"))
providers.to_csv(OUTPUT / "cricos_search.csv", index=False)
providers[["providerCode", "institutionType", "state", "postcode", "city",
           "institutionCapacity", "courseCount", "activeCourseCount"]]

## 2. Look up two providers by code

`lookup` takes 1 to 20 provider codes. A code that is not in the register is listed in `RUN_SUMMARY` as `notFound`
and is not charged. The two codes here come from the search above.

In [ ]:
codes = list(providers["providerCode"].head(2))

# Free tier: 2 rows x $0.002 = $0.004
lookup_run = run_actor(
    ACTOR,
    {"mode": "lookup", "providerCodes": codes},
    max_charge_usd=0.03,
)
looked_up = pd.DataFrame(client.dataset(lookup_run.default_dataset_id).iterate_items())
looked_up[["providerCode", "institutionType", "state", "city", "institutionCapacity", "courseCount", "activeCourseCount"]]

## Notes

* Contains data from the Commonwealth Register of Institutions and Courses for Overseas Students (CRICOS), Department of Education, Australian Government, licensed under Creative Commons Attribution 2.5 Australia (CC BY 2.5 AU). This Actor filters and reshapes the data; changes were made.
* A dated copy of two register files; the register itself is the authority. Do not use these rows to decide whether a provider may enrol a student.
* Names are left out for providers the name rule cannot confirm as organisations, and the website is left out with them. Postcode and city come from the provider's postal address, which may differ from where it teaches.
* Rows belong to the Actor's data terms, not to this repository's MIT licence. The CSV is written to `output/`, which git ignores.